# WR Air Yards Debt — Phase 0 POC (Jaxon Smith-Njigba, 2025)

**Hypothesis:** Books anchor WR receiving yards props to recent *actual* yards.
When a WR's actual yards << intended air yards (overthrows, drops, tight coverage),
next-game prop is under-priced. Test this on JSN (NFL receiving yards leader 2025, 1,793 yds).

**Steps:**
- 0a: Pull JSN per-game receiving stats from nfl-data-py (actual yards, intended air yards, targets)
- 0b: Pull JSN receiving yards props from Odds API (historical, 2025 season)
- 0c: Baseline RMSE/MAE — how well does the prop line predict actuals?
- 0d: Air yards debt scatter — does high debt → more yards next game?

**Decision gate:** If air_yards_debt has no correlation with next-game yards → revisit hypothesis before Phase 1.

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import requests
from dotenv import load_dotenv

# CWD is already repo root (os.chdir run above) — use Path(".") not Path("..")
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

# ── Config ────────────────────────────────────────────────────────────────────
SEASON          = 2025          # nfl-data-py uses integer season year
PLAYER_NAME_PBP = "J.Smith-Njigba"   # format used in nfl-data-py pbp data
PLAYER_NAME_API = "Jaxon Smith-Njigba"  # format used in Odds API

ODDS_API_KEY    = os.environ.get("ODDS_API_KEY", "")
ODDS_API_BASE   = "https://api.the-odds-api.com/v4"
SPORT           = "americanfootball_nfl"
MARKET          = "player_reception_yds"   # confirmed in config/the-odds-api_config.yaml
BOOKMAKERS      = ["draftkings", "fanduel", "betmgm", "williamhill_us"]

DATA_DIR = REPO_ROOT / "data" / "nfl" / "props"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root : {REPO_ROOT}")
print(f"Odds API key set: {bool(ODDS_API_KEY)}")

Repo root : /Users/thomasmyles/dev/betting/notebooks
Odds API key set: True


## Step 0a — Pull JSN Game Logs from nfl-data-py

For each game: actual receiving yards, intended air yards (all targets), completed air yards, YAC, targets.

`air_yards_debt = intended_air_yards - actual_rec_yards`
High debt = WR got deep looks but didn't convert (overthrows, drops, PBUs).

In [2]:
import nfl_data_py as nfl

# Pull full play-by-play for 2025 season
# Columns we need: game_id, week, posteam, receiver_player_name, receiver_player_id,
#                  air_yards, yards_gained, complete_pass, play_type
pbp_cols = [
    "game_id", "week", "season", "posteam", "defteam",
    "play_type", "pass_attempt",
    "receiver_player_name", "receiver_player_id",
    "air_yards",        # air yards on this target (all attempts, complete or not)
    "yards_gained",     # actual yards gained on the play (0 if incomplete)
    "complete_pass",    # 1 = completion, 0 = incompletion
    "yards_after_catch",# YAC (null if incomplete)
    "home_team", "away_team",
]

print("Fetching play-by-play data...")
pbp_raw = nfl.import_pbp_data([SEASON], columns=pbp_cols)
print(f"Raw PBP rows: {len(pbp_raw):,}")

# Filter to pass plays with a target (pass_attempt == 1 captures both complete + incomplete)
pbp_passes = pbp_raw[
    (pbp_raw["play_type"] == "pass") &
    (pbp_raw["pass_attempt"] == 1) &
    (pbp_raw["receiver_player_name"].notna())
].copy()

print(f"Pass plays with a target: {len(pbp_passes):,}")
print(f"Unique receivers: {pbp_passes['receiver_player_name'].nunique():,}")

Fetching play-by-play data...
2025 done.
Downcasting floats.
Raw PBP rows: 48,771
Pass plays with a target: 17,579
Unique receivers: 499


In [3]:
# Filter to JSN only
jsn_plays = pbp_passes[pbp_passes["receiver_player_name"] == PLAYER_NAME_PBP].copy()
print(f"JSN targets in 2025: {len(jsn_plays):,}")
print(f"Games: {jsn_plays['game_id'].nunique()}")
print(jsn_plays[["game_id", "week", "air_yards", "yards_gained", "complete_pass"]].head(10))

JSN targets in 2025: 189
Games: 20
             game_id  week  air_yards  yards_gained  complete_pass
2216  2025_01_SF_SEA     1       13.0          18.0            1.0
2242  2025_01_SF_SEA     1       19.0          21.0            1.0
2260  2025_01_SF_SEA     1       18.0           0.0            0.0
2287  2025_01_SF_SEA     1       17.0          22.0            1.0
2289  2025_01_SF_SEA     1       21.0           0.0            0.0
2327  2025_01_SF_SEA     1       16.0          16.0            1.0
2330  2025_01_SF_SEA     1       -3.0           2.0            1.0
2345  2025_01_SF_SEA     1       -8.0          -8.0            1.0
2346  2025_01_SF_SEA     1       27.0           0.0            0.0
2347  2025_01_SF_SEA     1        1.0           2.0            1.0


In [4]:
# Aggregate to per-game stats
jsn_games = (
    jsn_plays
    .groupby(["game_id", "week", "posteam", "defteam", "home_team", "away_team"])
    .agg(
        targets              = ("air_yards",        "count"),
        receptions           = ("complete_pass",    "sum"),
        actual_rec_yards     = ("yards_gained",     "sum"),
        intended_air_yards   = ("air_yards",        "sum"),   # ALL targets (complete + incomplete)
        rec_air_yards        = ("air_yards",        lambda x: (x * jsn_plays.loc[x.index, "complete_pass"]).sum()),
        yac                  = ("yards_after_catch","sum"),
    )
    .reset_index()
    .sort_values("week")
)

# Derived metrics
jsn_games["adot"]           = jsn_games["intended_air_yards"] / jsn_games["targets"]
jsn_games["air_yards_debt"] = jsn_games["intended_air_yards"] - jsn_games["actual_rec_yards"]
jsn_games["catch_rate"]     = jsn_games["receptions"] / jsn_games["targets"]
jsn_games["home_flag"]      = (jsn_games["posteam"] == jsn_games["home_team"]).astype(int)

# Next-game actual yards (what we're trying to predict / what the prop targets)
jsn_games["next_game_actual_yards"] = jsn_games["actual_rec_yards"].shift(-1)

print(f"Games in dataset: {len(jsn_games)}")
print(f"Season total yards: {jsn_games['actual_rec_yards'].sum():.0f}")
jsn_games[[
    "week", "targets", "receptions", "actual_rec_yards",
    "intended_air_yards", "air_yards_debt", "adot", "catch_rate"
]].round(1)

Games in dataset: 20
Season total yards: 1992


,week,targets,receptions,actual_rec_yards,intended_air_yards,air_yards_debt,adot,catch_rate
0,1,13,9.0,124.0,178.0,54.0,13.7,0.7
1,2,10,8.0,103.0,103.0,0.0,10.3,0.8
2,3,6,5.0,96.0,87.0,-9.0,14.5,0.8
3,4,5,4.0,79.0,78.0,-1.0,15.6,0.8
4,5,9,8.0,132.0,106.0,-26.0,11.8,0.9
5,6,13,8.0,162.0,176.0,14.0,13.5,0.6
6,7,14,8.0,123.0,167.0,44.0,11.9,0.6
7,9,9,8.0,129.0,94.0,-35.0,10.4,0.9
8,10,6,5.0,93.0,102.0,9.0,17.0,0.8
9,11,12,9.0,105.0,117.0,12.0,9.8,0.8


## Step 0b — Pull JSN Receiving Yards Props from Odds API

Fetch historical prop lines for each JSN game from the Odds API.
We need: game event ID → prop line + over/under odds for each bookmaker → consensus line.

In [5]:
import nfl_data_py as nfl

# Use nfl-data-py schedules instead of Odds API events endpoint.
# The /sports/{sport}/events endpoint only returns upcoming games —
# for a completed season we need schedules from nfl-data-py.
schedules = nfl.import_schedules([SEASON])

# Filter to SEA games (home or away)
TEAM_ABR      = "SEA"
TEAM_ODDS_API = "Seattle Seahawks"

jsn_events = schedules[
    (schedules["home_team"] == TEAM_ABR) |
    (schedules["away_team"] == TEAM_ABR)
].copy()

# Keep only regular season + playoffs (drop preseason week < 1)
jsn_events = jsn_events[jsn_events["week"] >= 1].copy()

# Normalise columns we need downstream
jsn_events = jsn_events.rename(columns={"game_id": "nfl_game_id"})
jsn_events["commence_time"] = pd.to_datetime(jsn_events["gameday"])
jsn_events = jsn_events.sort_values("week").reset_index(drop=True)

print(f"SEA games in 2025 schedule: {len(jsn_events)}")
jsn_events[["week", "nfl_game_id", "commence_time", "home_team", "away_team"]].head(10)

SEA games in 2025 schedule: 20


,week,nfl_game_id,commence_time,home_team,away_team
0,1,2025_01_SF_SEA,2025-09-07,SEA,SF
1,2,2025_02_SEA_PIT,2025-09-14,PIT,SEA
2,3,2025_03_NO_SEA,2025-09-21,SEA,NO
3,4,2025_04_SEA_ARI,2025-09-25,ARI,SEA
4,5,2025_05_TB_SEA,2025-10-05,SEA,TB
5,6,2025_06_SEA_JAX,2025-10-12,JAX,SEA
6,7,2025_07_HOU_SEA,2025-10-20,SEA,HOU
7,9,2025_09_SEA_WAS,2025-11-02,WAS,SEA
8,10,2025_10_ARI_SEA,2025-11-09,SEA,ARI
9,11,2025_11_SEA_LA,2025-11-16,LA,SEA


In [6]:
# jsn_events is already filtered to SEA from the schedules fetch above.
# Odds API uses full team names — build a lookup for the props fetch.
# nfl-data-py abbr "SEA" → Odds API "Seattle Seahawks" (set in TEAM_ODDS_API above)
print(f"SEA games to fetch props for: {len(jsn_events)}")
print(f"Date range: {jsn_events['commence_time'].min().date()} → {jsn_events['commence_time'].max().date()}")
jsn_events[["week", "commence_time", "home_team", "away_team"]].head(5)

SEA games to fetch props for: 20
Date range: 2025-09-07 → 2026-02-08


,week,commence_time,home_team,away_team
0,1,2025-09-07,SEA,SF
1,2,2025-09-14,PIT,SEA
2,3,2025-09-21,SEA,NO
3,4,2025-09-25,ARI,SEA
4,5,2025-10-05,SEA,TB


In [7]:
def fetch_player_prop_by_date(api_key: str, game_date: str, market: str, bookmakers: list,
                              home_team: str, away_team: str) -> list[dict]:
    """
    Fetch historical player props for a specific game via the Odds API historical endpoint.
    game_date: "YYYY-MM-DD"
    Costs 1 request per call. Returns list of {bookmaker, player_name, side, line, odds}.
    """
    # Historical odds endpoint — snapshots odds at a point in time
    # Use game date at noon ET (17:00 UTC) as a pre-game snapshot
    snapshot_time = f"{game_date}T17:00:00Z"
    url = f"{ODDS_API_BASE}/historical/sports/{SPORT}/odds"

    params = {
        "apiKey":     api_key,
        "markets":    market,
        "bookmakers": ",".join(bookmakers),
        "oddsFormat": "american",
        "dateFormat": "iso",
        "date":       snapshot_time,
    }

    resp = requests.get(url, params=params, timeout=30)
    if resp.status_code in (404, 422):
        return []
    resp.raise_for_status()

    data = resp.json()
    events = data.get("data", []) if isinstance(data, dict) else data

    rows = []
    for event in events:
        # Match to our specific game by team names
        h = event.get("home_team", "")
        a = event.get("away_team", "")
        if home_team not in (h, a) and away_team not in (h, a):
            continue

        event_id = event.get("id", "")
        for bookie in event.get("bookmakers", []):
            for mkt in bookie.get("markets", []):
                if mkt["key"] != market:
                    continue
                for outcome in mkt.get("outcomes", []):
                    rows.append({
                        "event_id":    event_id,
                        "bookmaker":   bookie["key"],
                        "player_name": outcome.get("description", ""),
                        "side":        outcome["name"],
                        "line":        outcome.get("point", None),
                        "odds":        outcome["price"],
                    })
    return rows


def parse_prop_rows(rows: list[dict], player_name: str) -> pd.DataFrame:
    """Filter to target player, pivot Over/Under into one row per bookmaker."""
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame()

    # Match on last name (handles "J.Smith-Njigba" vs "Jaxon Smith-Njigba")
    last_name = player_name.split()[-1].lower()
    df = df[df["player_name"].str.lower().str.contains(last_name, na=False)]
    if df.empty:
        return pd.DataFrame()

    df_over  = df[df["side"] == "Over" ][["event_id", "bookmaker", "player_name", "line", "odds"]].rename(
        columns={"odds": "over_odds", "line": "line_over"})
    df_under = df[df["side"] == "Under"][["bookmaker", "player_name", "odds"]].rename(
        columns={"odds": "under_odds"})

    merged = df_over.merge(df_under, on=["bookmaker", "player_name"], how="inner")
    merged = merged.rename(columns={"line_over": "line"})
    return merged


print("Functions defined. Ready to fetch props per game.")

Functions defined. Ready to fetch props per game.


In [8]:
import time

PROPS_CACHE = DATA_DIR / "jsn_2025_rec_yards_props_raw.csv"

if PROPS_CACHE.exists():
    print(f"Loading cached props from {PROPS_CACHE}")
    props_all = pd.read_csv(PROPS_CACHE)
else:
    print(f"Fetching props for {len(jsn_events)} SEA games...")
    records = []

    for _, ev in jsn_events.iterrows():
        game_date  = ev["commence_time"].strftime("%Y-%m-%d")
        home_full  = TEAM_ODDS_API if ev["home_team"] == TEAM_ABR else ev["away_team"]
        away_full  = TEAM_ODDS_API if ev["away_team"] == TEAM_ABR else ev["home_team"]

        rows = fetch_player_prop_by_date(
            ODDS_API_KEY, game_date, MARKET, BOOKMAKERS,
            home_team=TEAM_ODDS_API, away_team=away_full,
        )
        parsed = parse_prop_rows(rows, PLAYER_NAME_API)

        if not parsed.empty:
            parsed["week"]         = ev["week"]
            parsed["game_date"]    = game_date
            parsed["nfl_game_id"]  = ev["nfl_game_id"]
            records.append(parsed)
            print(f"  Wk{int(ev['week']):>2} {game_date} | {len(parsed)} rows for JSN")
        else:
            print(f"  Wk{int(ev['week']):>2} {game_date} | no JSN prop found")

        time.sleep(0.3)

    props_all = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    props_all.to_csv(PROPS_CACHE, index=False)
    print(f"\nSaved to {PROPS_CACHE}")

print(f"\nTotal prop rows: {len(props_all)}")
props_all.head()

Loading cached props from /Users/thomasmyles/dev/betting/notebooks/data/nfl/props/jsn_2025_rec_yards_props_raw.csv


EmptyDataError: No columns to parse from file

In [ ]:
# Convert American odds → implied probability, remove vig
def american_to_decimal(american: float) -> float:
    if american > 0:
        return (american / 100) + 1
    return (100 / abs(american)) + 1


def implied_prob_no_vig(over_american: float, under_american: float) -> tuple[float, float]:
    """Vig-free implied probabilities from American odds."""
    p_over_raw  = 1 / american_to_decimal(over_american)
    p_under_raw = 1 / american_to_decimal(under_american)
    total = p_over_raw + p_under_raw
    return p_over_raw / total, p_under_raw / total


props_all["implied_p_over"], props_all["implied_p_under"] = zip(*props_all.apply(
    lambda r: implied_prob_no_vig(r["over_odds"], r["under_odds"]), axis=1
))

# Consensus line per game = median line across bookmakers
props_consensus = (
    props_all
    .groupby("event_id")
    .agg(
        consensus_line  = ("line",           "median"),
        implied_p_over  = ("implied_p_over",  "mean"),
        implied_p_under = ("implied_p_under", "mean"),
        n_books         = ("bookmaker",       "count"),
        game_date       = ("game_date",       "first"),
        nfl_game_id     = ("nfl_game_id",     "first"),
    )
    .reset_index()
)

print(f"Games with JSN prop: {len(props_consensus)}")
props_consensus[["game_date", "nfl_game_id", "consensus_line", "implied_p_over", "n_books"]].round(3)

## Step 0c — Baseline RMSE / MAE

Join actuals to prop lines and measure how well the consensus line predicts actual yards.
This is the bar any model needs to beat.

In [ ]:
# Join nfl-data-py game log to Odds API props.
# Both sources share the same game_id format (e.g. "2025_01_SF_SEA"):
#   jsn_games["game_id"]          — from nfl-data-py PBP groupby
#   props_consensus["nfl_game_id"] — carried through from import_schedules
df = jsn_games.merge(
    props_consensus[["nfl_game_id", "consensus_line", "implied_p_over", "implied_p_under"]],
    left_on="game_id",
    right_on="nfl_game_id",
    how="inner",
)

print(f"Matched games: {len(df)}")

# ── Baseline metrics ──────────────────────────────────────────────────────────
df["residual"] = df["actual_rec_yards"] - df["consensus_line"]

mae  = df["residual"].abs().mean()
rmse = np.sqrt((df["residual"] ** 2).mean())
bias = df["residual"].mean()
hit_over = (df["actual_rec_yards"] > df["consensus_line"]).mean()

print(f"\n── Book line baseline ───────────────────────────────")
print(f"  MAE:      {mae:.1f} yards")
print(f"  RMSE:     {rmse:.1f} yards")
print(f"  Bias:     {bias:+.1f} yards  ({'book sets line low' if bias > 0 else 'book sets line high'})")
print(f"  Hit over: {hit_over:.1%}  (50% = fair)")
print(f"─────────────────────────────────────────────────────")
print(f"\nThis is the bar. Any model must beat RMSE={rmse:.1f} on held-out games.")

## Step 0d — Air Yards Debt Signal Check

If `air_yards_debt` (this game) is positively correlated with `actual_rec_yards` (next game),
the signal exists. Flat or negative → revisit hypothesis.

In [ ]:
# Scatter: air_yards_debt (game N) vs actual_rec_yards (game N+1)
# drop last row (no next game)
df_signal = df.dropna(subset=["next_game_actual_yards"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("JSN 2025 — Air Yards Debt Signal Check", fontsize=13, fontweight="bold")

# ── Left: air_yards_debt → next game yards ────────────────────────────────────
ax = axes[0]
ax.scatter(df_signal["air_yards_debt"], df_signal["next_game_actual_yards"],
           color="#1a6faf", edgecolors="white", s=90, zorder=3)

# Regression line
slope, intercept, r, p_val, se = stats.linregress(
    df_signal["air_yards_debt"], df_signal["next_game_actual_yards"]
)
x_range = np.linspace(df_signal["air_yards_debt"].min(), df_signal["air_yards_debt"].max(), 100)
ax.plot(x_range, intercept + slope * x_range, color="#e05c1a", linewidth=2, zorder=4)
ax.axhline(df_signal["next_game_actual_yards"].mean(), color="grey", linestyle="--", linewidth=1)
ax.axvline(0, color="grey", linestyle="--", linewidth=1, alpha=0.5)

ax.set_xlabel("Air Yards Debt (game N)", fontsize=11)
ax.set_ylabel("Actual Rec Yards (game N+1)", fontsize=11)
ax.set_title(f"r={r:.2f}  p={p_val:.3f}  slope={slope:.2f}", fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate each point with week number
for _, row in df_signal.iterrows():
    ax.annotate(f"Wk{int(row['week'])}", (row["air_yards_debt"], row["next_game_actual_yards"]),
                textcoords="offset points", xytext=(4, 2), fontsize=7, color="grey")

# ── Right: residual vs air_yards_debt ─────────────────────────────────────────
# residual = actual_yards - consensus_line (does debt explain what the book missed?)
ax2 = axes[1]
ax2.scatter(df_signal["air_yards_debt"], df_signal["residual"],
            color="#2ca05a", edgecolors="white", s=90, zorder=3)

slope2, intercept2, r2, p2, _ = stats.linregress(
    df_signal["air_yards_debt"], df_signal["residual"]
)
ax2.plot(x_range, intercept2 + slope2 * x_range, color="#e05c1a", linewidth=2, zorder=4)
ax2.axhline(0, color="grey", linestyle="--", linewidth=1.5)

ax2.set_xlabel("Air Yards Debt (game N)", fontsize=11)
ax2.set_ylabel("Residual vs Prop Line (game N+1)\nActual − Consensus", fontsize=11)
ax2.set_title(f"r={r2:.2f}  p={p2:.3f}  (does debt predict what book missed?)", fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "jsn_air_yards_debt_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nLeft panel:  air_yards_debt → next-game yards    r={r:.2f}  p={p_val:.3f}")
print(f"Right panel: air_yards_debt → residual vs line  r={r2:.2f}  p={p2:.3f}")
print(f"\nRight panel is the money shot: r>0 means debt predicts what the book underpriced.")

In [ ]:
# Supporting distributions: actual yards vs prop line by week
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle("JSN 2025 — Actual Yards vs Prop Line by Week", fontsize=13, fontweight="bold")

weeks = df["week"].values
ax = axes[0]
ax.bar(weeks, df["actual_rec_yards"], label="Actual yards", color="#1a6faf", alpha=0.8, width=0.4, align="center")
ax.bar(weeks + 0.4, df["consensus_line"], label="Consensus prop line", color="#e05c1a", alpha=0.8, width=0.4, align="center")
ax.set_ylabel("Receiving yards")
ax.set_xlabel("Week")
ax.legend()
ax.grid(axis="y", alpha=0.3)

ax2 = axes[1]
ax2.bar(weeks, df["air_yards_debt"], color=["#1a6faf" if v >= 0 else "#c0392b" for v in df["air_yards_debt"]], alpha=0.8)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_ylabel("Air Yards Debt\n(intended − actual)")
ax2.set_xlabel("Week")
ax2.set_title("Positive = intended > actual (WR got looks but didn't convert)")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "jsn_weekly_yards_debt.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Decision gate summary ──────────────────────────────────────────────────────
print("=" * 58)
print("  PHASE 0 DECISION GATE")
print("=" * 58)
print(f"\n  Games in dataset:      {len(df)}")
print(f"  Games with next-game:  {len(df_signal)}")
print()
print(f"  Baseline (book line):")
print(f"    MAE  = {mae:.1f} yds")
print(f"    RMSE = {rmse:.1f} yds")
print(f"    Bias = {bias:+.1f} yds")
print()
print(f"  Air yards debt signal:")
print(f"    debt → next-game yards     r = {r:.2f}  p = {p_val:.3f}")
print(f"    debt → residual vs line    r = {r2:.2f}  p = {p2:.3f}")
print()

if r2 > 0.10 and p2 < 0.20:
    verdict = "✅ PROCEED to Phase 1 — signal present, worth scaling to all WRs"
elif r2 > 0 and p2 < 0.40:
    verdict = "⚠️  WEAK signal — proceed cautiously, validate on 2-3 other top WRs first"
else:
    verdict = "❌ NO signal for JSN — revisit hypothesis before Phase 1"

print(f"  Verdict: {verdict}")
print("=" * 58)
print()
print("  Next step if proceeding: Phase 1 — build full WR feature universe")
print("  (2022-25, all WRs with prop lines available)")